In [1]:
import os
import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [5]:
from deconveil.dds import deconveil_fit
from deconveil.inference import Inference
from deconveil.default_inference import DefInference
from deconveil.utils_fit import *
from deconveil.utils_processing import *
from deconveil.utils_plot import *
from deconveil import deconveil_fit
from deconveil.ds import deconveil_stats

In [7]:
def run_pydeseq2(rna_counts, metadata, output_path, design="~condition", alpha=0.05):
    """
    Runs PyDESeq2 analysis and saves the results.

    Parameters:
        rna_counts (pd.DataFrame): Count matrix with genes as rows and samples as columns.
        metadata (pd.DataFrame): Metadata for the samples with design factors.
        output_path (str): Directory to save the results.
        design_factors (str): Column in metadata to use for design.
        alpha (float): Significance level for statistical tests.
    """
    os.makedirs(output_path, exist_ok=True)
    
    # Initialize DESeq2 analysis
    inference = DefaultInference(n_cpus=8)
    dds = DeseqDataSet(
        counts=rna_counts,
        metadata=metadata,
        design_factors=None,  # compare samples based on condition
        refit_cooks=True,
        inference=inference,
    )

    # Fit DESeq2 model
    dds.fit_size_factors()
    dds.fit_genewise_dispersions()
    dds.fit_dispersion_trend()
    dds.fit_dispersion_prior()
    dds.fit_MAP_dispersions()
    dds.fit_LFC()
    dds.calculate_cooks()
    
    if dds.refit_cooks:
        dds.refit()

    # Perform statistical analysis
    stat_res_pydeseq = DeseqStats(dds, 
                                  alpha=alpha, 
                                  cooks_filter=True, 
                                  independent_filter=True, 
                                  contrast=["condition", "B", "A"])
    stat_res_pydeseq.run_wald_test()

    if stat_res_pydeseq.cooks_filter:
        stat_res_pydeseq._cooks_filtering()
    stat_res_pydeseq.p_values

    if stat_res_pydeseq.independent_filter:
        stat_res_pydeseq._independent_filtering()
    else:
        stat_res_pydeseq._p_value_adjustment()

    # Log-fold change shrinkage
    stat_res_pydeseq.lfc_shrink(coeff="condition[T.B]")
    stat_res_pydeseq.summary()

    # Save results
    results_path = os.path.join(output_path, f"{replicate_num}_res_CNnaive_{n_samples}_{n_genes}.csv")
    stat_res_pydeseq.results_df.to_csv(results_path)
    return(stat_res_pydeseq.results_df)


def run_deconveil(rna_counts, metadata, cnv, output_path, design="~condition", alpha=0.05):
    """
    Runs DeConveil analysis and saves the results.

    Parameters:
        rna_counts (pd.DataFrame): Count matrix with genes as rows and samples as columns.
        metadata (pd.DataFrame): Metadata for the samples with design factors.
        cnv (pd.DataFrame): Copy number variation (CNV) data matrix  with genes as rows and samples as columns.
        output_path (str): Directory to save the results.
        design_factors (str): Column in metadata to use for design.
        alpha (float): Significance level for statistical tests.
    """
    os.makedirs(output_path, exist_ok=True)
    
    # Initialize DeConveil inference
    inference = DefInference(n_cpus=8)

    # Fit DeConveil model
    dds = deconveil_fit(
        counts=rna_counts,
        metadata=metadata,
        cnv=cnv,
        design_factors="condition",
        inference=inference,
        refit_cooks=True
    )
    dds.fit_size_factors()
    dds.fit_genewise_dispersions()
    dds.fit_dispersion_trend()
    dds.fit_dispersion_prior()
    dds.fit_MAP_dispersions()
    dds.fit_LFC()
    dds.calculate_cooks()

    if dds.refit_cooks:
        dds.refit()  # Replace outlier counts

    # Statistical analysis
    stat_res_deconveil = deconveil_stats(
        dds, 
        alpha=alpha, 
        contrast=["condition", "B", "A"],
        independent_filter=True, 
        cooks_filter=True
    )
    stat_res_deconveil.run_wald_test()

    if stat_res_deconveil.independent_filter:
        stat_res_deconveil._independent_filtering()
    else:
        stat_res_deconveil._p_value_adjustment()

    # Log-fold change shrinkage
    stat_res_deconveil.lfc_shrink(coeff="condition[T.B]")
    stat_res_deconveil.summary()

    # Save results
    results_path = os.path.join(output_path, f"{replicate_num}_res_CNaware_{n_samples}_{n_genes}.csv")
    stat_res_deconveil.results_df.to_csv(results_path)
    return(stat_res_deconveil.results_df)

In [9]:
sample_sizes = [10,20,40,60]
gene_counts = [1000]
num_replicates = 10

In [15]:
def process_and_save_replicates(replicate_num, n_samples, n_genes):
    try:
        # Construct dynamic file paths for each replicate
        base_path = "/Users/katsiarynadavydzenka/Documents/PhD_AI/deconveilCaseStudies/simulations/data/simulations_2/replicates"
        rna_path = f"{base_path}/{replicate_num}_rna_cn_{n_samples}_{n_genes}.csv"
        cnv_path = f"{base_path}/{replicate_num}_cn_join_{n_samples}_{n_genes}.csv"
        metadata_path = f"{base_path}/{replicate_num}_metadata_{n_samples}_{n_genes}.csv"
        
        # Load data
        rna = pd.read_csv(rna_path, index_col=0)
        rna = rna.T
        cnv = pd.read_csv(cnv_path, index_col=0)
        metadata = pd.read_csv(metadata_path, index_col=0)
        cnv = cnv.T
        cnv = (cnv * 2).astype(int)
        
        # Run Pydeseq 
        pydeseq_output_path = "/Users/katsiarynadavydzenka/Documents/PhD_AI/deconveilCaseStudies/simulations/results/simulation_2/replicates_pydeseq/"
        res_pydeseq = run_pydeseq2(rna, metadata, pydeseq_output_path)

        # Run Deconveil 
        deconveil_output_path = "/Users/katsiarynadavydzenka/Documents/PhD_AI/deconveilCaseStudies/simulations/results/simulation_2/replicates_deconveil/"
        res_deconveil = run_deconveil(rna, metadata, cnv, deconveil_output_path)
        
        print(f"Processed replicate {replicate_num} for {n_samples} samples and {n_genes} genes.")
    except Exception as e:
        print(f"Error processing replicate {replicate_num} for {n_samples} samples and {n_genes}: {e}")

In [17]:
# Iterate over all combinations
for replicate_num in range(1, num_replicates + 1):
    for n_samples in sample_sizes:
        for n_genes in gene_counts:
            process_and_save_replicates(replicate_num, n_samples, n_genes)


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.19 seconds.

Fitting dispersion trend curve...
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.20 seconds.

Fitting LFCs...
... done in 0.11 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 4 outlier genes.

Fitting dispersions...
... done in 0.00 seconds.

Fitting MAP dispersions...
... done in 0.00 seconds.

Fitting LFCs...
... done in 0.00 seconds.

Running Wald tests...
... done in 0.07 seconds.

Fitting MAP LFCs...
... done in 0.13 seconds.



Log2 fold change & Wald test p-value: condition B vs A
           baseMean  log2FoldChange     lfcSE      stat        pvalue  \
g1     1.097372e+03        3.115238  0.443189  7.340618  2.126098e-13   
g2     2.743869e+03        1.115495  0.444121  2.989312  2.796066e-03   
g3     2.272342e+06        1.141899  0.413361  3.296271  9.797756e-04   
g4     4.402327e+03        1.666519  0.365638  4.923924  8.482594e-07   
g5     2.799665e+04        1.927060  0.916864  2.943658  3.243579e-03   
...             ...             ...       ...       ...           ...   
g996   7.158258e+02       -0.460931  0.263120 -1.948122  5.140041e-02   
g997   3.901071e+03       -0.423982  0.431004 -1.297644  1.944099e-01   
g998   5.766475e+02       -0.173440  0.368066 -0.592994  5.531851e-01   
g999   5.386731e+02       -0.032855  0.365740 -0.116275  9.074349e-01   
g1000  1.264436e+02       -0.475361  0.392949 -1.515108  1.297452e-01   

               padj  
g1     1.635460e-11  
g2     1.536300e-02  
g3

/var/folders/mh/vvy722dj18x7xj41n2qbdzs80000gn/T/ipykernel_41591/2532259931.py:81: DeprecationWarning: design_factors are deprecated and will soon be removedPlease consider providing a formulaic formula using the design argument instead
  dds = deconveil_fit(
Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.15 seconds.

Fitting dispersion trend curve...
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.19 seconds.

Fitting LFCs...
... done in 0.12 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 2 outlier genes.

Fitting dispersions...
... done in 0.00 seconds.

Fitting MAP dispersions...
... done in 0.00 seconds.

Fitting LFCs...
... done in 0.00 seconds.

Running Wald tests...


...

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.16 seconds.

Fitting dispersion trend curve...
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.23 seconds.

Fitting LFCs...
... done in 0.17 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.07 seconds.

Fitting MAP LFCs...
... done in 0.14 seconds.



Log2 fold change & Wald test p-value: condition B vs A
           baseMean  log2FoldChange     lfcSE       stat        pvalue  \
g1     5.753246e+02        1.955120  0.170317  11.647567  2.361032e-31   
g2     4.163439e+03        1.921727  0.151900  12.787111  1.935364e-37   
g3     2.392231e+06        1.483409  0.157354   9.703552  2.911826e-22   
g4     4.253451e+03        0.946897  0.159266   6.103102  1.040291e-09   
g5     2.468352e+04        2.359326  0.412699   6.080624  1.197159e-09   
...             ...             ...       ...        ...           ...   
g996   8.488411e+02       -0.084748  0.108256  -0.803230  4.218416e-01   
g997   3.796421e+03       -0.323486  0.169036  -2.018308  4.355923e-02   
g998   6.646525e+02       -0.333683  0.145530  -2.385765  1.704362e-02   
g999   6.265589e+02       -0.222090  0.153337  -1.516998  1.292672e-01   
g1000  1.734791e+02       -0.141487  0.149380  -0.989832  3.222560e-01   

               padj  
g1     4.372281e-30  
g2     4.720

/var/folders/mh/vvy722dj18x7xj41n2qbdzs80000gn/T/ipykernel_41591/2532259931.py:81: DeprecationWarning: design_factors are deprecated and will soon be removedPlease consider providing a formulaic formula using the design argument instead
  dds = deconveil_fit(
Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.12 seconds.

Fitting dispersion trend curve...
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.29 seconds.

Fitting LFCs...
... done in 0.12 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: condition B vs A
           baseMean  log2FoldChange     lfcSE      stat        pvalue  \
g1     5.753246e+02        1.451104  0.170547  8.778546  1.656013e-18   
g2     4.163439e+03        1.373571  0.151602  9.397655  5.579272e-21   
g3     2.392231e+06        2.508242  0.054420  6.305859  2.865995e-10   
g4     4.253451e+03        0.455592  0.157724  3.255957  1.130108e-03   
g5     2.468352e+04        2.723319  0.369271  4.582386  4.596996e-06   
...             ...             ...       ...       ...           ...   
g996   8.488411e+02       -0.036277  0.108231 -0.331142  7.405376e-01   
g997   3.796421e+03       -0.278959  0.170433 -1.941538  5.219310e-02   
g998   6.646525e+02       -0.300035  0.145349 -2.219268  2.646851e-02   
g999   6.265589e+02       -0.186408  0.153073 -1.347596  1.777885e-01   
g1000  1.734791e+02       -0.107156  0.149082 -0.783958  4.330647e-01   

               padj  
g1     1.577156e-17  
g2     6.563849e-20  
g3

... done in 0.07 seconds.

Fitting MAP LFCs...
... done in 0.13 seconds.

